# This is show case when usage changing dramaticaly 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import boxcox
from scipy.special import inv_boxcox
from pmdarima import auto_arima

# 1. DATA EXTRACTION (Estimated from your dashboard image)
dates = pd.date_range(start="2025-04-03", periods=37, freq="W-THU")
# Values in thousands (estimated visually)
values = [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    5,
    40,
    20,
    20,
    30,
    30,
    10,
    20,
    40,
    120,
    100,
    140,
    190,
    170,
    240,
    480,
    320,
    550,
    1450,
    1850,
    2250,
    1800,
    1700,
    2300,
    1400,
    1650,
    1400,
    900,
    1250,
    350,
]
df = pd.DataFrame({"date": dates, "executions": np.array(values) * 1000})
df.set_index("date", inplace=True)

# 2. CREATE EXOGENOUS VARIABLES
# Level Shift: Permanent jump starting Sept 25, 2025
df["level_shift"] = (df.index >= "2025-09-25").astype(int)

# Holiday Pulse: Temporary dip for the week of Dec 11
df["holiday_dip"] = (df.index == "2025-12-11").astype(int)

# 3. APPLY BOX-COX TRANSFORMATION
# This stabilizes the variance so the model isn't overwhelmed by the 2M+ peaks
data_transformed, lam = boxcox(df["executions"])
df["executions_bc"] = data_transformed

# 4. FIT AUTO_ARIMA WITH EXOG
# We pass the exogenous dummies so ARIMA doesn't treat the jump as "noise"
exog_train = df[["level_shift", "holiday_dip"]]

model = auto_arima(
    df["executions_bc"],
    exog=exog_train,
    seasonal=True,
    m=52,  # Weekly seasonality
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore",
)

print(f"Optimal Lambda (Box-Cox): {lam:.4f}")
print(model.summary())

# 5. FORECAST FOR JANUARY 2026 (Next 4 weeks)
forecast_steps = 4
# We must define future exog: Level shift remains 1, Holiday dip returns to 0
future_exog = pd.DataFrame(
    {"level_shift": [1] * forecast_steps, "holiday_dip": [0] * forecast_steps},
    index=pd.date_range(start="2025-12-18", periods=forecast_steps, freq="W-THU"),
)

forecast_bc = model.predict(n_periods=forecast_steps, exog=future_exog)

# 6. INVERSE BOX-COX (Back to original scale)
forecast_final = inv_boxcox(forecast_bc, lam)

print("\nForecasted Test Executions for Jan 2026:")
print(pd.Series(forecast_final, index=future_exog.index))